## Installing Packages in Jupyter Notebook
To install packages that will work in this notebook, run the installation command in a code cell using `!pip install`:

In [ ]:
import cv2
print("✅ OpenCV loaded successfully!", cv2.__version__)


In [ ]:
import os

os.makedirs("data/videos", exist_ok=True)
os.makedirs("data/frames", exist_ok=True)
os.makedirs("data/slates", exist_ok=True)

print("✅ Folder structure created:")
for folder in ["data/videos", "data/frames", "data/slates"]:
    print(" -", folder)


In [ ]:
import cv2, os

def extract_frames(video_path, output_dir, fps_interval=1):
    os.makedirs(output_dir, exist_ok=True)
    vidcap = cv2.VideoCapture(video_path)
    fps = int(vidcap.get(cv2.CAP_PROP_FPS))
    frame_count = 0
    success, image = vidcap.read()

    while success:
        current_frame = int(vidcap.get(cv2.CAP_PROP_POS_FRAMES))
        if current_frame % (fps * fps_interval) == 0:
            frame_filename = os.path.join(output_dir, f"frame_{frame_count:04d}.jpg")
            cv2.imwrite(frame_filename, image)
            frame_count += 1
        success, image = vidcap.read()

    vidcap.release()
    print(f"✅ Extracted {frame_count} frames from {video_path}")

# Run the function
extract_frames("data/videos/test3.mp4", "data/frames", fps_interval=1)


In [ ]:
from ultralytics import YOLO

# Load YOLOv8 model (nano for small dataset)
model = YOLO("yolov8n.pt")

# Train
model.train(
    data="data/clapperboard/data.yaml",  # path to your Roboflow dataset
    epochs=50,
    imgsz=640,
    batch=8
)


In [ ]:
import os
from inference_sdk import InferenceHTTPClient

client = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key="rcBADaYI1ccHYxPqau24"
)

WORKSPACE = "snapsync"
WORKFLOW_ID = "custom-workflow-2"

# Test one frame
abs_path = os.path.abspath("data/frames/test1/frame_0001.jpg")
result = client.run_workflow(
    workspace_name=WORKSPACE,
    workflow_id=WORKFLOW_ID,
    images={"image": abs_path},
    use_cache=True
)

# Inspect the structure
import json
print(json.dumps(result, indent=4))


In [ ]:
import os
import cv2
from inference_sdk import InferenceHTTPClient
import numpy as np

# --- CONFIG ---
ROBOFLOW_API_KEY = "rcBADaYI1ccHYxPqau24"
WORKSPACE = "snapsync"
WORKFLOW_ID = "custom-workflow-2"

FRAME_PATH = "data/frames/test3/frame_0169.jpg"
OUTPUT_DIR = "data/slates"

# --- Roboflow client ---
client = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key=ROBOFLOW_API_KEY
)

# --- Main processing ---
def process_frame(frame_path, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)
    img = cv2.imread(frame_path)
    abs_path = os.path.abspath(frame_path)
    h_img, w_img = img.shape[:2]

    # Run Roboflow workflow
    result = client.run_workflow(
        workspace_name=WORKSPACE,
        workflow_id=WORKFLOW_ID,
        images={"image": abs_path},
        use_cache=True
    )

    cropped_images = {}

    for item in result:
        preds_list = item.get("predictions", {}).get("predictions", [])
        for idx, pred in enumerate(preds_list):
            label = pred.get("class", "")
            # Save everything for now
            x, y, w, h = float(pred["x"]), float(pred["y"]), float(pred["width"]), float(pred["height"])
            x1, y1 = max(0, int(x - w/2)), max(0, int(y - h/2))
            x2, y2 = min(w_img, int(x + w/2)), min(h_img, int(y + h/2))

            crop = img[y1:y2, x1:x2]
            cropped_images[f"{label}_{idx}"] = crop

            # Save crop
            cv2.imwrite(os.path.join(output_dir, f"{label}_{idx}.jpg"), crop)

            # Draw rectangle on original image
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

        # --- Create side-by-side preview in one window ---
        if cropped_images:
            previews = []
            for label, crop in cropped_images.items():
                # Resize crop to same height as original for easy concatenation
                crop_h, crop_w = crop.shape[:2]
                scale_factor = h_img / crop_h
                crop_resized = cv2.resize(crop, (int(crop_w * scale_factor), h_img))
                previews.append(crop_resized)

            # Stack all crops horizontally with the original image at the left
            combined = np.hstack([img] + previews)

            cv2.imshow("Detections Preview", combined)
            cv2.waitKey(0)
            cv2.destroyAllWindows()

    print(f"Cropped images saved for: {list(cropped_images.keys())}")
    return cropped_images

# --- Example usage ---
process_frame(FRAME_PATH)
